# Asistente RAG de Notebooks personales

In [ ]:
# imports
import os
import glob
from dotenv import load_dotenv
import gradio as gr
import openai

In [ ]:
# imports de langchain, plotly y Chroma

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
#from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.prompts import PromptTemplate
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

In [ ]:
# Se utiliza un modelo de bajo costo.
MODEL = "gpt-4o-mini"
db_name = "vector_db"
root_dir ="C:\\Users\\sp314-52-51k3\\projects\\llm_engineering"

In [ ]:
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

### OBTENCIÓN DE CHUNKS DEL PRIMER TIPO: secciones de cada notebook.

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional #PARA QUE ES liST? 


@dataclass
class SectionChunk:
    chunk_id: str
    notebook_id: str
    section_title: str
    cell_start: int
    cell_end: int
    markdown_text: str
    code_text: str
    combined_text: str
    notebook_path: str
    week: Optional[str] = None
    topic: Optional[str] = None


@dataclass
class NotebookRecord:
    notebook_id: str
    name: str
    path: str
    week: Optional[str]
    topic: Optional[str]
    total_cells: int
    sections: List[SectionChunk] = field(default_factory=list)



In [ ]:
from pathlib import Path
from typing import List


def find_notebooks(root_dir: str) -> List[Path]:
    root = Path(root_dir)
    return sorted(root.rglob("*.ipynb"))

In [ ]:
# json sirve para leer archivos .ipynb, porque internamente son JSON.
import json

# dataclass nos permite crear estructuras de datos limpias y legibles.
from dataclasses import dataclass, field

# Path sirve para manejar rutas de archivos y carpetas de manera cómoda.
from pathlib import Path

from typing import Any, Dict, List, Optional, Tuple


@dataclass
class SectionChunk:
    # Identificador único del chunk de sección.
    chunk_id: str

    # Identificador del notebook al que pertenece esta sección.
    notebook_id: str

    # Título de la sección, por ejemplo: "Carga de datos".
    section_title: str

    # Índice de la primera celda incluida en la sección.
    cell_start: int

    # Índice de la última celda incluida en la sección.
    cell_end: int

    # Texto markdown acumulado dentro de esta sección.
    markdown_text: str

    # Código acumulado dentro de esta sección.
    code_text: str

    # Texto combinado que luego será útil para embeddings o búsquedas.
    #  junta título + markdown + código.
    combined_text: str

    # Ruta completa del notebook.
    notebook_path: str

    # Metadatos opcionales extraídos de la ruta.
    week: Optional[str] = None
    topic: Optional[str] = None


@dataclass
class NotebookRecord:
    # Identificador del notebook.
    notebook_id: str

    # Nombre del archivo, por ejemplo "mi_notebook.ipynb".
    name: str

    # Ruta completa del notebook.
    path: str

    # Semana o carpeta principal detectada desde la ruta.
    week: Optional[str]

    # Tema o subcarpeta detectada desde la ruta.
    topic: Optional[str]

    # Número total de celdas del notebook.
    total_cells: int

    # Lista de secciones detectadas dentro del notebook.
    sections: List[SectionChunk] = field(default_factory=list)


# 1) OBTENER NOTEBOOKS EN CARPETAS Y SUBCARPETAS

def find_notebooks(root_dir: str) -> List[Path]:
    """
    Busca todos los archivos .ipynb dentro de root_dir y sus subcarpetas.

    Parámetros:
        root_dir: ruta de la carpeta raíz donde están tus notebooks.

    Regresa:
        Una lista de objetos Path, ordenados.
    """
    notebook_paths = []
    
    # Convertimos la ruta a objeto Path para trabajar mejor con ella.
    root = Path(root_dir)
    for path in root.rglob("*.ipynb"):
        if len(path.relative_to(root).parts) == 2:
            notebook_paths.append(path)

    # rglob("*.ipynb") busca de forma recursiva todos los notebooks.
    # sorted(...) solo los ordena para tener resultados estables.
    return sorted(notebook_paths)



In [ ]:
# 2) LEER UN NOTEBOOK
def load_notebook(path: Path) -> Dict[str, Any]:
    """
    Abre un notebook .ipynb y lo carga como diccionario Python.

    Parámetros:
        path: ruta del notebook.

    Regresa:
        El contenido JSON convertido a diccionario.
        este diccionario contiene  cells', 'metadata', 'nbformat', 'nbformat_minor'
        a su vez cells es una lista de celdas, cada celda es un diccionario definido por cell_type, id, metadata, source
    """
    # Abrimos el archivo en modo lectura con codificación UTF-8.
    with path.open("r", encoding="utf-8") as f:
        # json.load convierte el contenido JSON en un diccionario Python.
        return json.load(f)



In [ ]:
# 3) NORMALIZAR CELDAS

def normalize_cells(nb_json: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Toma el JSON bruto del notebook y lo transforma en una lista de celdas
    con una estructura simple y uniforme.

    Parámetros:
        nb_json: contenido del notebook como diccionario.

    Regresa:
        Lista de diccionarios, uno por celda. Cada diccionario trae source en formato bonito, cell_type e index
    """
    # Aquí iremos guardando las celdas ya limpiadas.
    normalized = []

    # nb_json.get("cells", []) obtiene la lista de celdas, supongo que enumerada
    # Si no existe "cells", usa lista vacía para evitar errores.
    for idx, cell in enumerate(nb_json.get("cells", [])): #nb.get("cells") es una lista de celdas
        # source es el contenido principal de la celda.
        source = cell.get("source", [])
        # En notebooks, source a veces viene como lista de líneas.
        # Ejemplo: ["print('hola')\n", "print('mundo')"]
        # Queremos convertirlo a un solo string.
        if isinstance(source, list):
            source = "".join(source)

        # Agregamos una versión simplificada de la celda.
        normalized.append({
            # Índice de la celda en el notebook.
            "index": idx,

            # Tipo de celda: "markdown" o "code".
            "cell_type": cell.get("cell_type", ""),

            # Texto fuente limpio.
            "source": source.strip(),
        })

    return normalized 

In [ ]:
# 4) EXTRAER METADATOS DESDE LA RUTA

def extract_path_metadata(notebook_path: Path, root_dir: str) -> Tuple[Optional[str], Optional[str]]:
    """
    Extrae metadatos simples desde la ruta del archivo.

    Ejemplo de estructura esperada:
        root/semana_01/transformers/notebook.ipynb

    En ese caso:
        week = "semana_01"
        topic = "transformers"

    Parámetros:
        notebook_path: ruta completa del notebook.
        root_dir: ruta raíz del proyecto.

    Regresa:
        Tupla (week, topic)
    """
    # relative_to(root_dir) quita la parte común inicial.
    # parts convierte la ruta restante en tupla de segmentos.
    relative_parts = notebook_path.relative_to(root_dir).parts
    print(relative_parts)

    # Si hay al menos dos niveles, el primero después de root será week.
    week = relative_parts[0] if len(relative_parts) >= 2 else None

    # Si hay al menos tres niveles, el segundo será topic.
    topic = relative_parts[1] if len(relative_parts) >= 2 else None #####lo modifiqué, que pasa cuando hay más niveles? será que se descartan? 
                                                                    #si es así esta super bien
    return week, topic

In [ ]:
# 5) DETECTAR SI UNA CELDA ES TÍTULO DE SECCIÓN

def is_section_header(cell: Dict[str, Any]) -> bool:
    """
    Determina si una celda markdown abre una sección.
    Asumimos que las secciones empiezan con uno o más #

    Parámetros:
        cell: diccionario de celda normalizada.

    Regresa:
        True si la celda es encabezado de sección, False si no.
    """
    return (
        # Debe ser markdown.
        cell["cell_type"] == "markdown"
        # Y su texto, quitando espacios iniciales, debe empezar con ###.
        and cell["source"].lstrip().startswith("#")
    ) #devuelve un valor de verdad para cada celda, en la prueba le daré una celda en específico


def clean_section_title(markdown_text: str) -> str:
    """
    Limpia el título de sección quitando los # iniciales y espacios.

    Ejemplo:
        "### Feature Engineering" -> "Feature Engineering"
    """
    return markdown_text.lstrip("#").strip()

In [ ]:
# 6) CONSTRUIR SECCIONES A PARTIR DE LAS CELDAS

def build_sections(
    cells: List[Dict[str, Any]],
    notebook_id: str,
    notebook_path: str,
    week: Optional[str],
    topic: Optional[str],
) -> List[SectionChunk]:
    """
    Agrupa celdas en secciones semánticas usando encabezados markdown #.

    Lógica:
    - Cada vez que ve un #, empieza una nueva sección.
    - Todo lo que sigue pertenece a esa sección hasta encontrar el siguiente #.
    - Si hay contenido antes del primer #, se guarda como sección "Untitled Section".

    Regresa:
        Lista de SectionChunk.
    """
    # Lista final de secciones detectadas.
    sections: List[SectionChunk] = []

    # Variables que representan la sección actual.
    current_title = "Untitled Section"
    current_start = 0

    # Aquí acumularemos el markdown y el código de la sección actual.
    markdown_parts: List[str] = []
    code_parts: List[str] = []

    def flush_section(end_index: int, section_idx: int) -> None:
        """
        Guarda la sección actual en la lista `sections`.

        Esta función interna usa las variables acumuladas actuales.
        """
        # nonlocal indica que queremos usar y modificar las variables
        # definidas en la función externa build_sections.
        nonlocal markdown_parts, code_parts, current_title, current_start

        # Juntamos todos los textos markdown de la sección actual.
        markdown_text = "\n\n".join(part for part in markdown_parts if part)

        # Juntamos todos los bloques de código.
        code_text = "\n\n".join(part for part in code_parts if part)

        # combined_text será útil luego para indexación o embeddings.
        # Incluimos título + markdown + código.
        combined_text = "\n\n".join(
            part for part in [current_title, markdown_text, code_text] if part
        )

        # Solo guardamos la sección si tiene contenido real.
        if markdown_text or code_text:
            sections.append(
                SectionChunk(
                    chunk_id=f"{notebook_id}_section_{section_idx}",
                    notebook_id=notebook_id,
                    section_title=current_title,
                    cell_start=current_start,
                    cell_end=end_index,
                    markdown_text=markdown_text,
                    code_text=code_text,
                    combined_text=combined_text,
                    notebook_path=notebook_path,
                    week=week,
                    topic=topic,
                )
            )

        # Una vez guardada la sección, vaciamos listas.
        markdown_parts = []
        code_parts = []

    # section_idx sirve para generar ids únicos por sección.
    section_idx = 0

    # Recorremos todas las celdas ya normalizadas.
    for cell in cells:
        # Si la celda es un encabezado de sección:
        if is_section_header(cell):
            # Si ya había contenido acumulado, primero guardamos la sección anterior.
            if markdown_parts or code_parts:
                flush_section(cell["index"] - 1, section_idx)
                section_idx += 1

            # Actualizamos datos de la nueva sección.
            current_title = clean_section_title(cell["source"])
            current_start = cell["index"]

            # Guardamos también el markdown del título dentro del bloque.
            markdown_parts.append(cell["source"])
            continue

        # Si no es encabezado, la agregamos según su tipo.
        if cell["cell_type"] == "markdown":
            markdown_parts.append(cell["source"])
        elif cell["cell_type"] == "code":
            code_parts.append(cell["source"])

    # Al terminar el recorrido, si quedó contenido pendiente, lo guardamos.
    if markdown_parts or code_parts:
        flush_section(cells[-1]["index"] if cells else 0, section_idx)

    return sections

In [ ]:
# 7) CONSTRUIR UN NOTEBOOKRECORD COMPLETO

def parse_notebook(path: Path, root_dir: str) -> NotebookRecord:
    """
    Procesa un notebook completo:
    - lo carga
    - normaliza sus celdas
    - extrae metadatos de ruta
    - construye secciones
    - regresa NotebookRecord
    """
    # Cargamos el notebook crudo.
    nb_json = load_notebook(path)

    # Convertimos celdas a formato uniforme.
    cells = normalize_cells(nb_json)

    # Extraemos week/topic desde la ruta.
    week, topic = extract_path_metadata(path, root_dir)

    # notebook_id: por ahora usamos el nombre sin extensión.
    notebook_id = path.stem

    # Construimos las secciones de este notebook.
    sections = build_sections(
        cells=cells,
        notebook_id=notebook_id,
        notebook_path=str(path),
        week=week,
        topic=topic,
    )

    # Regresamos el registro global del notebook.
    return NotebookRecord(
        notebook_id=notebook_id,
        name=path.name,
        path=str(path),
        week=week,
        topic=topic,
        total_cells=len(cells),
        sections=sections,
    )


# 8) PROCESAR TODOS LOS NOTEBOOKS DE UNA CARPETA RAÍZ

def ingest_notebooks(root_dir: str) -> List[NotebookRecord]:
    """
    Procesa todos los notebooks encontrados dentro de root_dir.

    Regresa:
        Lista de NotebookRecord
    """
    # Buscamos todos los notebooks.
    notebook_paths = find_notebooks(root_dir)

    # Aquí acumularemos los resultados.
    records = []

    for path in notebook_paths:
        try:
            # Intentamos parsear cada notebook.
            record = parse_notebook(path, root_dir)
            records.append(record)

        except Exception as e:
            # Si uno falla, lo reportamos pero no detenemos todo el proceso.
            print(f"Error procesando {path}: {e}")

    return records

In [ ]:
records = ingest_notebooks(root_dir)

In [ ]:
# Total de notebooks y secciones de notebooks
total_sections = sum(len(r.sections) for r in records)
len(records), total_sections

In [ ]:
#detección de secciones que no pasan el filtro de "buenas" (deben traer markdown, codigo y sobrepasar cierta longitud)
weak_sections = []

for record in records:
    for section in record.sections:
        text = section.combined_text.strip()
        if len(text) < 300:
            weak_sections.append((record.name, section.section_title, len(text)))

print("Weak sections:", len(weak_sections))
#print(weak_sections[:10])


##### FILTRADO DE SECCIONES QUE NO PASAN FILTRO DE MARKDOWN+CODIGO Y LONGITUD SUFICIENTE.

In [ ]:
# fUNCIÓN DE ELEGIBILIDAD DE SECCIONES:
#tienen que tener markdown y codigo y logitud mayor a 100
#o bien solo alguno de esos dos pero con longitud mayor a 300.
bad_sections = []
good_sections=[]
bad_sections_count=0
good_with_both=0
good_with_markdown_only=0
good_with_code_only=0

for record in records:
    for section in record.sections:
        text = section.combined_text.strip()
  
        has_markdown = bool(section.markdown_text and section.markdown_text.strip())
        has_code = bool(section.code_text and section.code_text.strip())
        has_combined = bool(section.combined_text and section.combined_text.strip())

        if has_markdown and has_code and len(text)>=100:
            good_with_both += 1
            good_sections.append(section)
        elif has_markdown and has_code==False and len(text)>=300:
            good_with_markdown_only += 1
            good_sections.append(section)
        elif has_code and has_markdown==False and len(text)>=300:
            good_with_code_only += 1
            good_sections.append(section)
        else:
            bad_sections_count += 1
            bad_sections.append(section)


In [ ]:
good_with_markdown_only, good_with_code_only, good_with_both

##### PARA GOOD_SECTIONS, CONSTRUYAMOS UNA ESTRUCTURA QUE SEA COMPATIBLE PARA CHROMA. DEBE TENER TEXTO/METADATA
Chroma le asignará un id y el embedding

Note que el formato que pide croma para guardar los vectores (aplicando los embeddings) es una lista del tipo Document() y cada elemento de la lista tiene un campo de "metada" que es un diccionario de datos y un campo de page_content que es string

In [ ]:
documents = []
for section in good_sections:
    page_content = f"Notebook: {section.notebook_id}\n"
    page_content += f"Semana: {section.week}\n"
    page_content += f"Sección: {section.section_title}\n"
    page_content+= section.combined_text
    doc = Document(
        page_content=page_content,
        metadata={
            "chunk_id": section.chunk_id,
            "notebook_id": section.notebook_id,
            "notebook_name": section.notebook_path.split("\\")[-1],
            "notebook_path": section.notebook_path,
            "section_title": section.section_title,
            "week": section.week,
            "topic": section.topic,
            "cell_start": section.cell_start,
            "cell_end": section.cell_end,
        }
    )
    documents.append(doc)

### SE CREA EL SEGUNDO TIPO DE CHUNKS: sumarizados de los notebooks y los guardamos como json para futuras consultas.

Estos sumarizados serán parte de la colección de "documents". 

Si ya se creo el archivo summaries_cache donde están los summaries de cada notebook, ya no vuelve a llamar al llm.

In [ ]:
def summarize_notebook(nb_json, week, topic): #función en donde se crea  un archivo tipo document de un notebook sumarizado con llm
    a=[]
   # print(nb_json)
    for cell in nb_json.get("cells"): #Para celdas
        source=cell.get("source")
        source_text= " ".join(source)
        source_text+= "\n"
        a.append(source_text)
        
    notebook_content=" ".join(a)    
    #print(f"notebook_content: {notebook_content}\ln")
    notebook_content_llm = openai.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                 {"role": "user", "content": notebook_content}
                ]
        )
    page_content = f"{week}\n"
    page_content += f"{topic}\n"
    page_content += notebook_content_llm.choices[0].message.content
    
    #print(f"Page_content: {page_content}")
    doc=Document(
                page_content=page_content,
                metadata={
                    "week": week,
                    "topic": topic,
                    #"notebook_path": section.notebook_path,
                    }
                )
    doc1={"page_content": page_content,
                "metadata":{
                    "week": week,
                    "topic": topic}
                }
    return doc, doc1

In [ ]:
# Genera para todos los notebooks, su respectivo chunk se sumarizado
system_prompt="Eres un asistente experto en interpretar código en python"
system_prompt+="El usuario te proporcionará un notebook en python con código y markdown y tu deberás proporcionar\
                un summary detallado de lo que el código hace, sus objetivos, identificar metodologías utilizadas.\
                Hazlo en un resumen menor a 8 mil caracteres."
SUMMARIES_CACHE = "summaries_cache.json"

def load_or_generate_summaries(root_dir, cache_path=SUMMARIES_CACHE):
    if os.path.exists(cache_path): #si archivo de summaries existe, entonces basta con mandar llamar al archivo
        print("El archivo de summaries existe")
        with open(cache_path, "r", encoding="utf-8") as f: #lectura
            summaries=json.load(f)
            document_notebooks=[]
            for key in summaries.keys():
                doc=Document(page_content=summaries[key].get("page_content"), metadata= summaries[key].get("metadata")) 
                document_notebooks.append(doc)
            return document_notebooks

    document_notebooks=[] #Lista de notebooks summarizados con formato para vectorizar
    summaries={} #diccionario de summarizados que se guarda en memoria.
    notebook_paths = find_notebooks(root_dir) #lista de paths a consultar
    print("El archivo de summaries no existe, se genera uno nuevo")
    for path in notebook_paths:
        try:
            nb_json = load_notebook(path)            
            cells = normalize_cells(nb_json) # Convertimos celdas a formato uniforme.
            week, topic = extract_path_metadata(path, root_dir) # Extraemos week/topic de cada ruta.
            doc, doc1=summarize_notebook(nb_json, week, topic) #mandamos a sumarizar con llm y con formato correcto para vectorizar
            document_notebooks.append(doc) #el summarizado se agrega a la lista documentos para langchain
            doc_id= week+"_"+topic
            summaries[doc_id] = doc1 #el summarizado se guarda en dict summaries.
        except Exception as e:
            # Si uno falla, lo reportamos pero no detenemos todo el proceso.
            print(f"Error procesando {path}: {e}")
        
    # Se guarda de manera local el dict summaries
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(summaries, f, ensure_ascii=False, indent=2)
    
    return document_notebooks

In [ ]:
document_notebooks=load_or_generate_summaries(root_dir, cache_path=SUMMARIES_CACHE)

### UNION DE LOS DOS TIPOS DE CHUNKS: DOCUMENTS + DOCUMENT_NOTEBOOKS

In [ ]:
documents= documents + document_notebooks

In [ ]:
len(documents)

### Crear la base de vectores en chroma y visualización

In [ ]:
# Coloca los fragmentos de datos en un almacén de vectores que asocie una incrustación de vectores con cada fragmento
# Chroma es una popular base de datos de vectores de código abierto basada en SQLLite

embeddings = OpenAIEmbeddings()

# Borra la base de datos si existe
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

# Crea el vectorstore
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore creado con {vectorstore._collection.count()} documents")

In [ ]:
vectorstore

In [ ]:
# Investiguemos los vectores

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"Hay {count:,} vectores con {dimensions:,} dimensiones en el almacén de vectores")

In [ ]:
# Trabajo previo
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_weeks = [metadata['week'] for metadata in metadatas]
#lista de colores para cada document
colors = [['blue', 'green', 'red', 'orange', 'black', 'pink', 'yellow', 'grey'][['week1', 'week2', 'week3', 'week4', 'week5', 'week6', 'week7', 'week8'].index(t)] for t in doc_weeks]

In [ ]:
# Reducir la dimensionalidad de los vectores a 2D usando t-SNE
# (incrustación de vecinos estocásticos distribuidos en t)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_weeks, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='Visualización 2D Chroma Vector Store',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# gráfica en 3D

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_weeks, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='Visualización 3D Chroma Vector Store',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

### FASE ULTIMA DE RAG: cadena de conversación con llm +almacen vectorial + memoria

Se personaliza el metodo de ConversationalRetrievalChain, dándole un prompt de sistema particular

In [ ]:
# create a new Chat with OpenAI
llm = ChatOpenAI(temperature=0.7, model_name=MODEL)

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG
retriever = vectorstore.as_retriever() #va y busca en chroma

#"context", "question", "chat_history" son nombres por convencion que la chain espera, context en particular es el concatenado 
# de los chunks que el retriever encontro en chroma.
# Prompt que inyecta la metadata junto con el contexto
custom_prompt = PromptTemplate(
    input_variables=["context", "question", "chat_history"],
    template="""Eres un asistente experto en los proyectos y notebooks de LLM Engineering del usuario.
                Usa el siguiente contexto recuperado para responder. Cada fragmento incluye el nombre del notebook, 
                semana y sección de donde proviene — úsalos para dar respuestas precisas.
                Contexto:
                {context}
                Historial de conversación:
                {chat_history}
                Pregunta: {question}
                Respuesta:"""
)

conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm,retriever=retriever,memory=memory,combine_docs_chain_kwargs={"prompt": custom_prompt})


### CHAT

In [ ]:
def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

RECORDAR BORRAR MEMORIA ANTES DE CORRER EL CHAT

In [ ]:
view = gr.ChatInterface(chat).launch()